# [PBT] 화장품 이커머스 ㅣ 12. 모델링용 전처리

## 이 노트북이 하는 일

**⑤ 모델링용 전처리**

> 묻는 것 : **모델에 넣을 수 있는 모양의 train / test 두 벌을 같은 손으로 만들었는가**

```
11  다변량 → 최종 X 6종 · 변환 지시            (output\11_최종X.csv)
                    ─────────────────────
                    12  층화 분할 → 로그 변환 → 더미 → 저장      ← 이 노트북
                    ─────────────────────
13  모델 구축 — fit_pipeline(encode=False, scale=False)
```

**전처리를 `fit_pipeline`에 맡기지 않는다.** `my_logit.fit_pipeline`의 인코딩·스케일링은
넘긴 데이터프레임 **한 벌에만** 적용되고, 밖으로 나오는 것은 `recipe_` 뿐이라
인코더·스케일러가 남지 않는다. `my_logit.predict`도 상태를 갖지 않아
`add_constant` 만 붙이고 `new_data` 를 그대로 넘긴다.
**test 를 같은 모양으로 만드는 일은 우리 몫이고, 그 자리가 여기다.**

## 여기서 하지 않는 것

| 안 하는 것 | 이유 |
|---|---|
| **스케일링** | `report_variables` 가 βstd = B × SD(x) 를 **원단위 적합에서 직접 계산**한다. 오즈비도 원단위라야 "1달러 오를 때"로 읽히고, 트리 계열은 애초에 불필요 |
| **이상치 처리** | 설계 §3-7 · `11` `#10-2`. `price` 의 큰 값은 오류가 아니라 비싼 상품이고, 진짜 오류(음수·0원)는 `01` 에서 이미 제거했다 |
| **결측 대체** | `brand_grp` 결측은 `03` 에서 '미상' 범주로 끝났다. 여기서는 **남은 결측이 0인지 확인만** 한다 |
| **구간화** | **결정 21번 — ⑥/⑦ 의 Box-Tidwell 뒤로 옮겼다.** 아래 「진입 전 확정된 것」 참조 |
| **`recipe` 적용 · `fix_linear`** | 로짓 선형성 검정 **뒤에** 쓰는 도구다 → ⑥/⑦ |
| **검증셋 분리** | LAB-13 이 교차검증을 ⑦ 에 둔다. train 안에서 `StratifiedKFold` |
| **모델 적합 · 성능 평가** | ⑬ 모델 구축 |

## 진입 전 확정된 것 — 결과를 보기 전에 정해져 있다

`claude/결정기록_9-1.md` 에 근거 원본이 있다.

| # | 규칙 | 내용 |
|---|---|---|
| 19 | 번호 체계 | ⑤ 전처리 = **`12`** · ⑥ 모델 구축 = **`13`**. 산출물 접두사 `12a`~`12j` |
| 20 | X 목록 | **`11_최종X.csv` 를 읽는다.** 손으로 타이핑하지 않는다. 6행이 아니면 멈춘다 |
| 21 | 구간화 | **⑤ 에서 하지 않는다.** 시점을 ⑥/⑦ 로 옮긴다 (아래 대응 규칙) |
| 22 | 민감도 | 모형 B(`category_grp` 포함) 설계행렬을 **같은 분할·같은 전처리로 여기서 함께** 만든다 |
| 23 | 기준 범주 | `brand_grp` → **`runail`** · `cart_timeband` → `심야` · `cart_weekday` → `월` |
| — | 분할 | `test_size=0.2` · `stratify=y` · `random_state=RANDOM_STATE`(3217) |
| — | VIF | 더미 전개 후 **점검만.** `reduce_vif` 자동 제거는 쓰지 않는다 (설계 §3-9) |

### 결정 21번 — 구간화 대응 규칙 (여기서 못 박고, 결과를 보고 바꾸지 않는다)

> ⑥/⑦ 의 Box-Tidwell(`my_logit.test_linear`)에서 `cart_product_view_cnt` 가 α = 0.05 에서
> 로짓 선형성 **위배로 판정되면 그때** 구간화 버전으로 교체한다.
> 경계는 `11` 이 고정한 `0 / 1 / 2 / 3 / 4+` 를 그대로 쓴다.
> 교재가 우선하는 처방(**제곱항**)을 먼저 시도하고, 그것으로 통과하면 구간화하지 않는다.
> **AUC 등 성능 지표는 이 판정에 쓰지 않는다.**

사유 넷 — ① 교재상 구간화는 로짓이 계단 모양일 때의 **처방**이지 전처리 기본값이 아니다

② 결정 12번 민감도 판정 대상인 핵심 X 3개 중 하나라 더미로 쪼개지면 "부호가 같은가" 를 판정할 규칙이 없다 

③ βstd = B × SD 는 연속형 전제라, 보고서 Ⅴ-2 영향력 순위표에서 빠진다 

④ 재사용할 경계가 곧 `0→1` 계단인데 그 계단은 **Y 비율을 보고 찾은 것**이다.

### 결정 23번 — 왜 기준 범주를 우리가 지정하는가

`my_prep.dummies` 는 `drop_first=True` 일 때 **`sorted(df[c].unique())[0]`** 을 뺀다
(`my_prep.py` L241). 실제 데이터에서 `brand_grp` 의 기본 기준은 **`bpw.style`** — 최소 규모대
브랜드다. 그대로 두면 ⑧ 의 오즈비 14개가 전부 "bpw.style 대비" 로 읽히고, 표준오차도 가장 크다.

> **기준 범주를 바꿔도 모형은 동일하다.** 적합값 · AUC · 다른 변수의 계수와 p 값이 전부 그대로고,
> 바뀌는 것은 더미 계수를 "무엇 대비" 로 읽느냐뿐이다. **성능을 보고 고르는 선택이 아니다.**

선정 규칙은 **Y 와 무관하다** — 표본 크기와 범주의 순서 정의만 쓴다.
`brand_grp` 은 *표본이 가장 큰 실브랜드*(잔여 범주 `미상`·`기타`는 후보에서 제외),
`cart_timeband`·`cart_weekday` 는 *순서 범주의 첫 값*.

## 🚨 누수 등급표 — 발표 때 이것만 보면 답할 수 있다

| 처리 | 등급 | 이유 | 이 노트북의 위치 |
|---|---|---|---|
| **층화 분할** | — | **가장 먼저 한다** | `#02` |
| 로그 변환 | 🟢 안전 | 각 행이 **자기 값만** 쓴다. train 통계를 참조하지 않는다 | `#04` |
| 더미 인코딩 | 🟡 주의 | 범주 목록이 전체 기준이면 약한 누수 → **train 열로 `reindex`** | `#05` |
| 스케일링 · 구간화 · 결측 대체 | 🔴 위험 | train 통계를 쓴다 → 반드시 분할 후, train 에서만 `fit` | **하지 않음** |

> 로그 변환이 순서상 분할 뒤에 오지만 **논리적으로는 누수가 아니다.** 분할 전에 해도 값이 같다.
> 순서를 지키는 것은 "분할이 가장 먼저" 라는 규율을 코드 순서로 증명해 두기 위해서다.

## 👥 역할 — 파일이 아니라 셀로 나눈다

⑤ 는 **직렬**이다. 분할 → 변환 → 인코딩이 앞 결과에 물려 있어 나눌 축이 없다.
억지로 나누면 중간 파일이 하나 더 생기고 그 파일이 곧 사고 지점이 된다
(`10a`·`10b` 를 나눌 수 있었던 것은 **병렬**이었기 때문이다).

| 셀 | 누가 |
|---|---|
| `#01`·`#02`·`#04`·`#05`·`#06`·`#07`·`#09` | **만드는 사람** |
| `#03` 분할 검증 · `#08` 검수 체크리스트 | **검수하는 사람** — 읽고 서명한다 |

## 📐 수치 표기 규칙 (설계 §4-5)

> 판정문의 모든 수치에 **어느 셀에서 나오는지**를 `#03` 형태로 적는다.
> 앞 노트북의 CSV 에서 온 것은 `11_최종X.csv` 처럼 파일을 밝힌다. **둘 다 아닌 수치는 쓰지 않는다.**

---

## #01. 준비

In [2]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)
sys.path.insert(0, BASE)

from pandas import read_csv, to_datetime, DataFrame, Series, set_option, concat
from sklearn.model_selection import train_test_split

from helpers import my_stats, my_prep, RANDOM_STATE

set_option("display.max_columns", 60)
set_option("display.max_colwidth", 70)

OUT  = BASE + r"\output"

TEST_SIZE = 0.2
VIF_MAX   = 10.0        # 설계 §3-9 — 점검용 눈금. 자동 제거는 하지 않는다

# 결정 23번 — 더미 기준 범주. 선정 규칙은 표본 크기와 순서 정의뿐이다 (Y 와 무관)
REF = {"brand_grp":     "runail",   # 표본 최대 실브랜드 (미상·기타는 잔여 범주라 후보 제외)
       "cart_timeband": "심야",     # 순서 범주 심야→오전→오후→저녁 의 첫 구간
       "cart_weekday":  "월",       # 순서 범주 월~일 의 첫 요일
       "category_grp":  "기타"}     # 모형 B 전용. 대조용이라 계수를 읽지 않으므로 최빈 범주

# 어떤 경우에도 X 에 들어가면 안 되는 것 (설계 §12 · 브리핑 §4)
FORBIDDEN = ["lag_hours", "purchase_7d", "user_id", "t0", "product_id",
             "brand", "category_id",
             "has_prior_view", "prior_uniq_product_cnt"]   # 11 다변량에서 제외됨

print("데이터 폴더 :", OUT)
print("RANDOM_STATE :", RANDOM_STATE)

데이터 폴더 : .\output
RANDOM_STATE : 3217


### 1-1. 마스터 테이블 + `08` 대조 — 하나라도 어긋나면 멈춘다

In [3]:
m = read_csv(OUT + r"\03_master.csv", encoding="utf-8-sig")

m["t0"]          = to_datetime(m["t0"], format="%Y-%m-%d %H:%M:%S")
m["product_id"]  = m["product_id"].astype("int32")
m["purchase_7d"] = m["purchase_7d"].astype("int8")
m["cart_timeband"] = m["cart_timeband"].astype("category").cat.set_categories(
    ["심야", "오전", "오후", "저녁"], ordered=True)
m["cart_weekday"]  = m["cart_weekday"].astype("category").cat.set_categories(
    ["월", "화", "수", "목", "금", "토", "일"], ordered=True)
for c in ["brand_grp", "category_grp"]:
    m[c] = m[c].astype("category")

ok = [("행 수",        f"{len(m):,}", "108,458"),
      ("1행 = 1고객",  str(len(m) == m["user_id"].nunique()), "True"),
      ("Y 양성",       f"{int(m['purchase_7d'].sum()):,}명 / {m['purchase_7d'].mean():.2%}",
                       "22,236명 / 20.50%"),
      ("brand_grp",    f"{m['brand_grp'].nunique()}종", "15종"),
      ("category_grp", f"{m['category_grp'].nunique()}종", "59종"),
      # lag_hours 는 Y=0 이면 결측이 정상이고, brand 는 '미상' 으로 채우기 전의 추적용 원본이다
      ("결측 (lag_hours·brand 제외)",
                       f"{int(m.drop(columns=['lag_hours', 'brand']).isna().sum().sum())}개",
                       "0개")]
chk = DataFrame(ok, columns=["항목", "이번 실행", "08번 값"])
chk["일치"] = chk["이번 실행"] == chk["08번 값"]

assert chk["일치"].all(), "❌ 08번과 어긋난다. 여기서 멈춘다."
chk

,항목,이번 실행,08번 값,일치
0,행 수,"108,458","108,458",True
1,1행 = 1고객,True,True,True
2,Y 양성,"22,236명 / 20.50%","22,236명 / 20.50%",True
3,brand_grp,15종,15종,True
4,category_grp,59종,59종,True
5,결측 (lag_hours·brand 제외),0개,0개,True


### 1-2. `11_최종X.csv` 수령 — **X 목록을 손으로 타이핑하지 않는다** (결정 20번)

`11` `#10-3` 이 *"⑤ 전처리 노트북이 읽어 이 목록대로만 처리한다"* 는 용도로 저장한 파일이다.
옮겨 적으면 그 연결이 끊기고, 두 곳이 어긋나도 오류가 나지 않는다.

In [4]:
TODO5   = read_csv(OUT + r"\11_최종X.csv",       encoding="utf-8-sig", index_col="변수")
LOGPLAN = read_csv(OUT + r"\11_로그변환판정.csv", encoding="utf-8-sig", index_col="변수")

FINAL_X = list(TODO5.index)
CONT3   = list(TODO5[TODO5["척도"] == "연속"].index)
NOM3    = list(TODO5[TODO5["척도"] == "명목"].index)

# 로그 변환 계획을 지시서에서 직접 뽑는다 — 방법(log / log1p)도 우리가 정하지 않는다
LOGS = {"log_columns":   [c for c in CONT3
                          if c in LOGPLAN.index and LOGPLAN.loc[c, "방법"] == "log"],
        "log1p_columns": [c for c in CONT3
                          if c in LOGPLAN.index and LOGPLAN.loc[c, "방법"] == "log1p"]}

가드 = [("11_최종X.csv 행 수", len(FINAL_X), 6),
        ("연속형",             len(CONT3),   3),
        ("명목형",             len(NOM3),    3),
        ("모두 채택 판정",     int((TODO5["판정"] == "채택").sum()), 6),
        ("마스터에 전부 존재", int(sum(c in m.columns for c in FINAL_X)), 6),
        ("기준 범주 지정 완료", int(sum(c in REF for c in NOM3)), 3)]
G = DataFrame(가드, columns=["항목", "이번 실행", "기대"])
G["일치"] = G["이번 실행"] == G["기대"]

assert G["일치"].all(), "❌ 11번 지시서와 어긋난다. 여기서 멈춘다."

print(f"최종 X {len(FINAL_X)}종 : {FINAL_X}")
print(f"  연속 {CONT3}")
print(f"  명목 {NOM3}")
print(f"로그 변환 : log{LOGS['log_columns']}  ·  log1p{LOGS['log1p_columns']}")
print()
print("⑤에서 할 일 (11_최종X.csv 원문) ─────────────────")
for c in FINAL_X:
    print(f"  {c:24s} {TODO5.loc[c, '⑤에서 할 일']}")
G

최종 X 6종 : ['price', 'prior_view_cnt', 'cart_product_view_cnt', 'brand_grp', 'cart_timeband', 'cart_weekday']
  연속 ['price', 'prior_view_cnt', 'cart_product_view_cnt']
  명목 ['brand_grp', 'cart_timeband', 'cart_weekday']
로그 변환 : log['price']  ·  log1p['prior_view_cnt', 'cart_product_view_cnt']

⑤에서 할 일 (11_최종X.csv 원문) ─────────────────
  price                    log 변환
  prior_view_cnt           log1p 변환
  cart_product_view_cnt    log1p 변환 · 구간화 검토 — 단조이나 0→1 계단이 전체 폭의 절반 이상 (#08-2)
  brand_grp                더미 인코딩 (분할 후)
  cart_timeband            더미 인코딩 (분할 후)
  cart_weekday             더미 인코딩 (분할 후)


,항목,이번 실행,기대,일치
0,11_최종X.csv 행 수,6,6,True
1,연속형,3,3,True
2,명목형,3,3,True
3,모두 채택 판정,6,6,True
4,마스터에 전부 존재,6,6,True
5,기준 범주 지정 완료,3,3,True


### 1-3. 누수 차단 — 금지 변수가 X 프레임에 들어갔는가

`lag_hours` 는 t₀ **이후** 정보이고 Y=1 일 때만 값이 있다. 넣는 순간 정답을 보고 맞히는 것이 된다.
`has_prior_view`·`prior_uniq_product_cnt` 는 `11` 다변량에서 제외된 변수다.

`category_grp` 은 **모형 A 의 X 가 아니다.** 결정 22번에 따라 모형 B 설계행렬을 같은 분할로
만들기 위해 **동승자로만** 태운다. A 의 설계행렬에는 들어가지 않으며 `#05-3` 에서 확인한다.

In [5]:
X = m[FINAL_X + ["category_grp"]].copy()      # category_grp = 동승자 (모형 B 전용)
y = m["purchase_7d"].copy()

X.index = m["user_id"].values                  # ⑥·⑧ 에서 행을 되짚을 수 있게
y.index = m["user_id"].values
X.index.name = y.index.name = "user_id"

침입 = [c for c in FORBIDDEN if c in X.columns]
assert not 침입, f"❌ 금지 변수가 X 에 있다: {침입}"

print(f"X {X.shape[0]:,}행 × {X.shape[1]}열   (모형 A 의 X 는 {len(FINAL_X)}종 + 동승자 category_grp)")
print(f"y {y.shape[0]:,}행   양성 {int(y.sum()):,}명 / {y.mean():.4%}")
print(f"금지 변수 {len(FORBIDDEN)}종 부재 확인 ✅  {FORBIDDEN}")
print(f"X 결측 : {int(X.isna().sum().sum())}개")
X.head(3)

X 108,458행 × 7열   (모형 A 의 X 는 6종 + 동승자 category_grp)
y 108,458행   양성 22,236명 / 20.5019%
금지 변수 9종 부재 확인 ✅  ['lag_hours', 'purchase_7d', 'user_id', 't0', 'product_id', 'brand', 'category_id', 'has_prior_view', 'prior_uniq_product_cnt']
X 결측 : 0개


,price,prior_view_cnt,cart_product_view_cnt,brand_grp,cart_timeband,cart_weekday,category_grp
user_id,,,,,,,
543079411,4.29,1,0,미상,심야,금,1487580005268456287
555391563,5.24,0,0,grattol,심야,금,1602943681873052386
385999154,13.02,1,0,italwax,심야,금,1487580013069861041


---

## #02. 층화 분할 — **가장 먼저 한다** 〈만드는 사람〉

`stratify=y` 가 하는 일 : Y=1 을 8:2 로, Y=0 을 8:2 로 **따로 나눠 합친다.**
그래서 두 쪽의 양성 비율이 전체와 같아진다.

쓰지 않으면 20.50% 가 우연히 19.8% / 22.9% 로 갈릴 수 있고,
**그 차이가 그대로 성능 차이로 보인다.** 오류는 나지 않는다 — 숫자만 조용히 틀린다.

교재 근거 — LAB-13 04 (층화 분할).

In [6]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,                    # ← 이 한 줄이 아래 #03 의 세 비율을 같게 만든다
    random_state=RANDOM_STATE)     # helpers 전역 상수 3217

print(f"train  {X_tr.shape[0]:,}행 × {X_tr.shape[1]}열")
print(f"test   {X_te.shape[0]:,}행 × {X_te.shape[1]}열")
print(f"합계   {X_tr.shape[0] + X_te.shape[0]:,}행  ==  원본 {len(X):,}행 : "
      f"{'✅' if X_tr.shape[0] + X_te.shape[0] == len(X) else '❌'}")
print(f"인덱스 교집합 : {len(set(X_tr.index) & set(X_te.index))}명  (0 이어야 한다)")

train  86,766행 × 7열
test   21,692행 × 7열
합계   108,458행  ==  원본 108,458행 : ✅
인덱스 교집합 : 0명  (0 이어야 한다)


## #03. 분할 검증

**소수 둘째 자리까지 같아야 한다.** 하나라도 어긋나면 `stratify` 가 걸리지 않은 것이다.

In [7]:
rows = []
for nm, s, 기대 in [("전체", y, len(m)), ("train", y_tr, 86766), ("test", y_te, 21692)]:
    rows.append({"구분": nm, "행 수": len(s), "기대 행 수": 기대,
                 "양성": int(s.sum()), "양성 비율(%)": round(s.mean() * 100, 4)})

SPLIT = DataFrame(rows).set_index("구분")
SPLIT["행 수 일치"] = SPLIT["행 수"] == SPLIT["기대 행 수"]
SPLIT["비율 일치(소수 둘째)"] = SPLIT["양성 비율(%)"].round(2) == round(y.mean() * 100, 2)

for nm, r in SPLIT.iterrows():
    print(f"{nm:6s} {r['행 수']:>7,}행   양성 {r['양성']:>6,}명   {r['양성 비율(%)']:.4f}%")
print()
print(f"세 비율이 소수 둘째 자리까지 같은가 : "
      f"{'✅ 같다' if SPLIT['비율 일치(소수 둘째)'].all() else '❌ 다르다 — 여기서 멈춘다'}")

assert SPLIT["비율 일치(소수 둘째)"].all(), "❌ 층화가 걸리지 않았다."
assert SPLIT["행 수 일치"].all(), "❌ 행 수가 기대와 다르다. test_size 나 원본을 확인할 것."
SPLIT

전체     108,458행   양성 22,236명   20.5019%
train   86,766행   양성 17,789명   20.5023%
test    21,692행   양성  4,447명   20.5006%

세 비율이 소수 둘째 자리까지 같은가 : ✅ 같다


,행 수,기대 행 수,양성,양성 비율(%),행 수 일치,비율 일치(소수 둘째)
구분,,,,,,
전체,108458,108458,22236,20.5019,True,True
train,86766,86766,17789,20.5023,True,True
test,21692,21692,4447,20.5006,True,True


---

## #04. 로그 변환 🟢 — 행 단위라 누수가 아니다

각 행이 **자기 값만** 쓴다. train 의 평균·표준편차 같은 통계를 참조하지 않으므로
순서상 분할 뒤에 오지만 **논리적으로는 누수가 아니다.** 분할 전에 해도 값이 같다.

방법(`log` / `log1p`)은 `11_로그변환판정.csv` 에서 읽어 온 것이다 — 여기서 정하지 않는다.
`price` 는 최소 \$0.05 로 0 을 포함하지 않아 `log`, 나머지 둘은 0 을 포함해 `log1p`.

In [8]:
전     = {c: float(X_tr[c].skew()) for c in CONT3}
원_고유 = {c: int(X_tr[c].nunique()) for c in CONT3}   # #08 구간화 여부 판정에 쓴다

X_tr = my_prep.log_transform(X_tr, **LOGS, verbose=True)
X_te = my_prep.log_transform(X_te, **LOGS, verbose=False)

후 = {c: float(X_tr[c].skew()) for c in CONT3}

SKEW = DataFrame({"방법":  [LOGPLAN.loc[c, "방법"] for c in CONT3],
                  "11 왜도(전체)": [LOGPLAN.loc[c, "왜도"] for c in CONT3],
                  "변환 전(train)": [round(전[c], 3) for c in CONT3],
                  "변환 후(train)": [round(후[c], 3) for c in CONT3]},
                 index=CONT3)
SKEW.index.name = "변수"
SKEW["감소폭"]  = (SKEW["변환 전(train)"] - SKEW["변환 후(train)"]).round(3)
SKEW["기준 1.0 이하"] = SKEW["변환 후(train)"].abs() <= 1.0

print()
print("※ prior_view_cnt 가 변환 후에도 1.0 을 넘는 것은 사전에 기록된 한계다 (결정 13번).")
print("   잔차 왜도는 조회 건수의 실제 긴 꼬리(최대 71회)에서 오며 단조변환으로 제거할 수 없다.")
print("   로짓 선형성은 ⑬ 의 Box-Tidwell 로 별도 확인한다.")
SKEW

컬럼        꼬리방향      변환식                   역변환식                                  왜도
----------------------------------------------------------------------------------------
price     우측 꼬리     log(x)                exp(y)                    7.25 -> 0.04
prior_view_cnt우측 꼬리     log(1+x)              exp(y)-1                  5.83 -> 1.04
cart_product_view_cnt우측 꼬리     log(1+x)              exp(y)-1                  1.75 -> 0.81

※ prior_view_cnt 가 변환 후에도 1.0 을 넘는 것은 사전에 기록된 한계다 (결정 13번).
   잔차 왜도는 조회 건수의 실제 긴 꼬리(최대 71회)에서 오며 단조변환으로 제거할 수 없다.
   로짓 선형성은 ⑬ 의 Box-Tidwell 로 별도 확인한다.


,방법,11 왜도(전체),변환 전(train),변환 후(train),감소폭,기준 1.0 이하
변수,,,,,,
price,log,7.21,7.246,0.041,7.205,True
prior_view_cnt,log1p,5.90,5.829,1.043,4.786,False
cart_product_view_cnt,log1p,1.75,1.745,0.811,0.934,True


---

## #05. 더미 인코딩 🟡 — 기준 범주를 **우리가 지정한다** (결정 23번)

### 5-1. helpers 기본값이 무엇을 기준으로 삼는가 — 먼저 눈으로 본다

`my_prep.dummies` 는 `drop_first=True` 일 때 `sorted(df[c].unique())[0]` 을 뺀다.
파이썬의 문자열 정렬은 코드포인트 순이라 **ASCII 가 한글보다 앞선다.**

In [9]:
rows = []
for c in NOM3 + ["category_grp"]:
    values  = sorted(X_tr[c].astype(str).unique())
    기본    = values[0]
    비율    = (X_tr[c].astype(str) == 기본).mean() * 100
    지정    = REF[c]
    비율2   = (X_tr[c].astype(str) == 지정).mean() * 100
    rows.append({"변수": c, "종수": len(values),
                 "helpers 기본 기준": 기본,   "기본 비중(%)": round(비율, 2),
                 "우리가 지정한 기준": 지정, "지정 비중(%)": round(비율2, 2),
                 "교체": 기본 != 지정})

REFTAB = DataFrame(rows).set_index("변수")
REFTAB["모형"] = ["A·B"] * len(NOM3) + ["B 전용"]

for c in NOM3 + ["category_grp"]:
    assert REF[c] in set(X_tr[c].astype(str).unique()), f"❌ {c} 의 기준 범주 {REF[c]} 가 train 에 없다"

print("brand_grp 정렬 앞머리 :", sorted(X_tr['brand_grp'].astype(str).unique())[:4], "...")
print()
print("→ 그대로 두면 ⑧ 의 오즈비 14개가 전부 'bpw.style 대비' 로 읽힌다.")
print("  최소 규모대 브랜드라 표준오차도 가장 크다. 그래서 교체한다.")
REFTAB

brand_grp 정렬 앞머리 : ['bpw.style', 'cnd', 'concept', 'estel'] ...

→ 그대로 두면 ⑧ 의 오즈비 14개가 전부 'bpw.style 대비' 로 읽힌다.
  최소 규모대 브랜드라 표준오차도 가장 크다. 그래서 교체한다.


,종수,helpers 기본 기준,기본 비중(%),우리가 지정한 기준,지정 비중(%),교체,모형
변수,,,,,,,
brand_grp,15,bpw.style,1.92,runail,6.24,True,A·B
cart_timeband,4,심야,8.45,심야,8.45,False,A·B
cart_weekday,7,금,16.82,월,14.36,True,A·B
category_grp,59,1487580004916134735,0.50,기타,32.77,True,B 전용


### 5-2. 설계행렬 만들기 — 전 범주를 편 뒤 기준 열을 직접 뺀다

`drop_first=True` 로는 기준을 지정할 수 없으므로 `drop_first=False` 로 전개한 뒤
기준 열 **하나만** 직접 제거한다. **결과 열 수는 `drop_first=True` 와 동일하다.**

> ⚠️ `my_prep.dummies` 는 값의 종류가 **2개인 컬럼을 변환 대상에서 제외**하고 원래 컬럼을 유지한다
> (`my_prep.py` L212). 우리 명목형은 15·4·7 종이라 해당되지 않지만, 아래에서 확인한다.

In [10]:
def make_design(df, nominals, refs, verbose=True):
    """전 범주 더미로 편 뒤 지정한 기준 열만 직접 제거한다. drop_first=True 와 열 수가 같다."""
    for c in nominals:
        assert df[c].nunique() > 2, f"❌ {c} 가 2종이라 dummies 가 건너뛴다"

    D = my_prep.dummies(df, columns=nominals, drop_first=False, verbose=verbose)

    dropped = []
    for c in nominals:
        col = f"{c}_{refs[c]}"
        assert col in D.columns, f"❌ 기준 열 {col} 이 만들어지지 않았다"
        D = D.drop(columns=[col])
        dropped.append(col)

    if verbose:
        print(f"\n기준 범주로 제거한 열 {len(dropped)}개 : {dropped}")
    return D, dropped


# ── 모형 A — category_grp 을 여기서 내린다
A_tr, ref_cols = make_design(X_tr.drop(columns=["category_grp"]), NOM3, REF, verbose=True)
A_te, _        = make_design(X_te.drop(columns=["category_grp"]), NOM3, REF, verbose=False)

print(f"\n모형 A : {A_tr.shape[1]}열   (연속 {len(CONT3)} + 더미 {A_tr.shape[1] - len(CONT3)})")

brand_grp (15종) -> 15개: brand_grp_bpw.style, brand_grp_cnd, brand_grp_concept, brand_grp_estel, brand_grp_grattol, brand_grp_ingarden, brand_grp_irisk, brand_grp_italwax, brand_grp_jessnail, brand_grp_kapous, brand_grp_masura, brand_grp_runail, brand_grp_uno, brand_grp_기타, brand_grp_미상
cart_timeband (4종) -> 4개: cart_timeband_심야, cart_timeband_오전, cart_timeband_오후, cart_timeband_저녁
cart_weekday (7종) -> 7개: cart_weekday_월, cart_weekday_화, cart_weekday_수, cart_weekday_목, cart_weekday_금, cart_weekday_토, cart_weekday_일

컬럼 수: 6개 -> 29개

기준 범주로 제거한 열 3개 : ['brand_grp_runail', 'cart_timeband_심야', 'cart_weekday_월']

모형 A : 26열   (연속 3 + 더미 23)


### 5-3. train 열을 기준으로 test 를 맞춘다 — 🟡 누수 차단의 핵심

`reindex` 로 **test 에만 있는 열은 버리고, 없는 열은 0 으로 채운다.**
범주 목록을 전체 기준으로 잡으면 약한 누수가 되므로 **train 이 기준**이다.

기대 열 수 : 연속 3 + `brand_grp` 14 + `cart_timeband` 3 + `cart_weekday` 6 = **26열**

In [11]:
A_te = A_te.reindex(columns=A_tr.columns, fill_value=0)

assert list(A_tr.columns) == list(A_te.columns), "❌ train / test 의 열이 순서까지 같지 않다"
assert "category_grp" not in "".join(A_tr.columns), "❌ category_grp 이 모형 A 에 섞였다"
침입A = [c for c in FORBIDDEN if c in A_tr.columns]
assert not 침입A, f"❌ 금지 변수가 설계행렬에 있다: {침입A}"

내역 = [("연속형", len(CONT3), len(CONT3))]
for c in NOM3:
    n = len([k for k in A_tr.columns if k.startswith(f"{c}_")])
    내역.append((c, n, m[c].nunique() - 1))
COLS = DataFrame(내역, columns=["구성", "실제 열 수", "기대"]).set_index("구성")
COLS["일치"] = COLS["실제 열 수"] == COLS["기대"]

print(f"모형 A : train {A_tr.shape[0]:,} × {A_tr.shape[1]}열  /  test {A_te.shape[0]:,} × {A_te.shape[1]}열")
print(f"열 순서까지 일치 ✅   ·   총 {A_tr.shape[1]}열 == 26 : {'✅' if A_tr.shape[1] == 26 else '❌'}")
print(f"EPV : train 기준 {int(y_tr.sum()):,}/{A_tr.shape[1]} = {round(int(y_tr.sum()) / A_tr.shape[1])}"
      f"   ·   전체 기준 {int(y.sum()):,}/{A_tr.shape[1]} = {round(int(y.sum()) / A_tr.shape[1])}"
      f"   (통상 기준 10)")
print(" ※ 브리핑 §4 의 'EPV 855' 는 전체 표본 기준이다. 실제 적합은 train 으로 하므로 684 가 맞는 값이다.")
print(f"결측 : train {int(A_tr.isna().sum().sum())}개 · test {int(A_te.isna().sum().sum())}개")
COLS

모형 A : train 86,766 × 26열  /  test 21,692 × 26열
열 순서까지 일치 ✅   ·   총 26열 == 26 : ✅
EPV : train 기준 17,789/26 = 684   ·   전체 기준 22,236/26 = 855   (통상 기준 10)
 ※ 브리핑 §4 의 'EPV 855' 는 전체 표본 기준이다. 실제 적합은 train 으로 하므로 684 가 맞는 값이다.
결측 : train 0개 · test 0개


,실제 열 수,기대,일치
구성,,,
연속형,3,3,True
brand_grp,14,14,True
cart_timeband,3,3,True
cart_weekday,6,6,True


---

## #06. 더미 전개 후 VIF — **점검만 한다** (설계 §3-9)

`11` `#03` 에서 못 본 부분이다. 명목형 VIF 는 더미로 펴야 계산되므로 여기가 첫 자리다.

**`my_prep.reduce_vif` 자동 제거는 쓰지 않는다.** 더미는 서로 VIF 가 조금 오르는 것이 정상이라
임계(10.0) 초과가 나와도 **판정은 사람이 한다.** `11` 에서 제외 후 최대 VIF 가 1.249 였으므로,
여기서 크게 오른다면 그것은 더미 구조에서 온 것이다.

In [12]:
VIF3 = my_stats.compute_vif(A_tr, columns=list(A_tr.columns))
VIF3["임계 초과"] = VIF3["VIF"] > VIF_MAX

초과 = VIF3[VIF3["임계 초과"]]
print(f"최대 VIF : {VIF3['VIF'].max():.3f}   (임계 {VIF_MAX})")
print(f"임계 초과 : {len(초과)}개" + (f"  → {list(초과.index)}" if len(초과) else "  ✅ 없음"))
print(f"연속 3종 VIF : " +
      " · ".join(f"{c} {VIF3.loc[c, 'VIF']:.3f}" for c in CONT3 if c in VIF3.index))
print()
print("※ 자동 제거하지 않는다. 초과가 있으면 #08 에서 사람이 판정하고 사유를 적는다.")
VIF3.head(12)

최대 VIF : 4.482   (임계 10.0)
임계 초과 : 0개  ✅ 없음
연속 3종 VIF : price 1.192 · prior_view_cnt 1.694 · cart_product_view_cnt 1.723

※ 자동 제거하지 않는다. 초과가 있으면 #08 에서 사람이 판정하고 사유를 적는다.


,VIF,임계 초과
brand_grp_미상,4.481510,False
brand_grp_기타,3.763870,False
cart_timeband_오후,3.374646,False
cart_timeband_저녁,3.305424,False
cart_timeband_오전,2.817174,False
cart_weekday_금,1.808128,False
cart_weekday_목,1.730787,False
cart_product_view_cnt,1.722728,False
brand_grp_irisk,1.721413,False
cart_weekday_토,1.699725,False


---

## #07. 민감도용 모형 B — 같은 분할 · 같은 전처리, **X 목록 한 줄만 다르다** (결정 22번)

결정 12번이 요구하는 조건이다. ⑬ 에서 B 를 새로 만들면 "같은 분할" 이 코드가 아니라
**말로만** 남는다. 그래서 여기서 같은 인덱스 · 같은 로그 변환을 거친 프레임으로 함께 만든다.

| 모형 | 열 수 | `category_grp` | 쓰임 |
|---|---|---|---|
| **A** | 26 | 제외 | 기본 모형 |
| **B** | 84 | 포함 (59종 → 58 더미) | 대조용. **핵심 X 3개의 부호·유의성만 본다** |

> **AUC 등 성능 지표는 이 판정에 쓰지 않는다.** 성능이 좋은 쪽을 고르는 순간 규율 위반이다.

In [13]:
NOM4 = NOM3 + ["category_grp"]

B_tr, ref_cols_B = make_design(X_tr, NOM4, REF, verbose=False)
B_te, _          = make_design(X_te, NOM4, REF, verbose=False)
B_te = B_te.reindex(columns=B_tr.columns, fill_value=0)

# "X 목록 한 줄만 다르다" 를 코드로 확인한다 — A 의 26열이 B 안에 그대로, 같은 값으로 있어야 한다
공통 = [c for c in A_tr.columns if c in B_tr.columns]
assert len(공통) == A_tr.shape[1], "❌ A 의 열이 B 에 그대로 들어 있지 않다"
assert A_tr[공통].equals(B_tr[공통]), "❌ A 와 B 의 공통 열 값이 다르다 — 전처리가 갈렸다"
assert list(A_tr.index) == list(B_tr.index), "❌ A 와 B 의 분할이 다르다"
assert list(A_tr.columns) == list(A_te.columns) == 공통

추가 = [c for c in B_tr.columns if c not in A_tr.columns]
print(f"모형 A : {A_tr.shape[1]}열   ·   모형 B : {B_tr.shape[1]}열   (차이 {len(추가)}열)")
print(f"B 가 26 + 58 = 84열인가 : {'✅' if B_tr.shape[1] == 84 else '❌ ' + str(B_tr.shape[1])}")
print(f"A 의 26열이 B 안에 같은 값으로 존재 ✅   ·   분할 인덱스 동일 ✅")
print(f"추가된 열은 전부 category_grp 더미인가 : "
      f"{'✅' if all(c.startswith('category_grp_') for c in 추가) else '❌'}")
print(f"EPV(B) : train 기준 {round(int(y_tr.sum()) / B_tr.shape[1])}"
      f"   ·   전체 기준 {round(int(y.sum()) / B_tr.shape[1])}   (브리핑 §6 의 265 는 전체 기준)")

모형 A : 26열   ·   모형 B : 84열   (차이 58열)
B 가 26 + 58 = 84열인가 : ✅
A 의 26열이 B 안에 같은 값으로 존재 ✅   ·   분할 인덱스 동일 ✅
추가된 열은 전부 category_grp 더미인가 : ✅
EPV(B) : train 기준 212   ·   전체 기준 265   (브리핑 §6 의 265 는 전체 기준)


---

## #08. 🔍 검수 체크리스트 — 〈검수하는 사람이 채운다〉

`10c` `#09` 를 둘이 함께 채운 것과 같은 방식이다. **만드는 손과 확인하는 눈을 나누되, 파일은 하나로.**

아래 8-1 은 **기계가 확인할 수 있는 것**이고, 8-2 는 **사람이 눈으로 보고 서명하는 것**이다.

### 8-1. 기계 확인 — 하나라도 ❌ 면 저장하지 않는다

In [14]:
검수 = []
def 확인(항목, 조건, 근거=""):
    검수.append({"항목": 항목, "결과": "✅" if bool(조건) else "❌", "근거": 근거})

확인("전체 / train / test 의 Y 비율이 소수 둘째 자리까지 같다",
     SPLIT["비율 일치(소수 둘째)"].all(),
     " / ".join(f"{r['양성 비율(%)']:.2f}%" for _, r in SPLIT.iterrows()) + "  (#03)")

확인("X_tr.columns 와 X_te.columns 가 순서까지 같다",
     list(A_tr.columns) == list(A_te.columns), f"{A_tr.shape[1]}열 (#05-3)")

확인("설계행렬이 26열이다",
     A_tr.shape[1] == 26, f"연속 {len(CONT3)} + 더미 {A_tr.shape[1] - len(CONT3)} (#05-3)")

확인("스케일링을 하지 않았다 (연속 3종의 평균 0 · 표준편차 1 이 아니다)",
     not all(abs(A_tr[c].mean()) < 1e-6 and abs(A_tr[c].std() - 1) < 1e-6 for c in CONT3),
     " · ".join(f"{c} 평균 {A_tr[c].mean():.3f}" for c in CONT3) + "  (#04)")

확인("결측 대체를 하지 않았다 (애초에 결측이 0)",
     int(A_tr.isna().sum().sum()) == 0 and int(A_te.isna().sum().sum()) == 0, "0개 (#05-3)")

확인("구간화를 하지 않았다 (로그 변환 전후로 고유값 개수가 그대로다 — 묶으면 줄어든다)",
     all(A_tr[c].nunique() == 원_고유[c] for c in CONT3),
     " · ".join(f"{c} {원_고유[c]}종→{A_tr[c].nunique()}종" for c in CONT3) + "  (결정 21번)")

확인("금지 변수가 설계행렬에 없다",
     not [c for c in FORBIDDEN if c in A_tr.columns], f"{FORBIDDEN} (#01-3)")

확인("category_grp 이 모형 A 에 없다",
     not [c for c in A_tr.columns if c.startswith("category_grp")], "(#05-3)")

확인("행 수가 86,766 / 21,692 / 86,766 / 21,692 이다",
     [len(A_tr), len(A_te), len(y_tr), len(y_te)] == [86766, 21692, 86766, 21692],
     f"{len(A_tr):,} / {len(A_te):,} / {len(y_tr):,} / {len(y_te):,}  (#02)")

확인("X 와 y 의 인덱스가 같은 순서다",
     list(A_tr.index) == list(y_tr.index) and list(A_te.index) == list(y_te.index),
     "user_id 기준 (#01-3)")

확인("기준 범주가 결정 23번대로다",
     ref_cols == [f"{c}_{REF[c]}" for c in NOM3], f"{ref_cols} (#05-2)")

확인("모형 B 가 84열이고 A 의 26열을 그대로 포함한다",
     B_tr.shape[1] == 84 and A_tr[공통].equals(B_tr[공통]),
     f"B {B_tr.shape[1]}열 (#07)")

확인(f"더미 후 VIF 임계 {VIF_MAX} 초과가 없다",
     not VIF3["임계 초과"].any(),
     f"최대 {VIF3['VIF'].max():.3f} (#06)")

CHECK = DataFrame(검수)
실패 = CHECK[CHECK["결과"] == "❌"]

print(f"{len(CHECK) - len(실패)} / {len(CHECK)} 통과")
if len(실패):
    print("\n❌ 아래 항목을 해결하기 전에는 #09 저장을 실행하지 않는다.")
    for _, r in 실패.iterrows():
        print(f"   - {r['항목']}  ({r['근거']})")
CHECK

13 / 13 통과


,항목,결과,근거
0,전체 / train / test 의 Y 비율이 소수 둘째 자리까지 같다,✅,20.50% / 20.50% / 20.50% (#03)
1,X_tr.columns 와 X_te.columns 가 순서까지 같다,✅,26열 (#05-3)
2,설계행렬이 26열이다,✅,연속 3 + 더미 23 (#05-3)
3,스케일링을 하지 않았다 (연속 3종의 평균 0 · 표준편차 1 이 아니다),✅,price 평균 1.606 · prior_view_cnt 평균 0.575 · cart_product_view_cnt 평...
4,결측 대체를 하지 않았다 (애초에 결측이 0),✅,0개 (#05-3)
5,구간화를 하지 않았다 (로그 변환 전후로 고유값 개수가 그대로다 — 묶으면 줄어든다),✅,price 1637종→1637종 · prior_view_cnt 49종→49종 · cart_product_view_cnt...
6,금지 변수가 설계행렬에 없다,✅,"['lag_hours', 'purchase_7d', 'user_id', 't0', 'product_id', 'brand..."
7,category_grp 이 모형 A 에 없다,✅,(#05-3)
8,"행 수가 86,766 / 21,692 / 86,766 / 21,692 이다",✅,"86,766 / 21,692 / 86,766 / 21,692 (#02)"
9,X 와 y 의 인덱스가 같은 순서다,✅,user_id 기준 (#01-3)


### 8-2. 사람 확인 — 검수하는 사람이 읽고 채운다

기계가 대신할 수 없는 것들이다. **셀을 위에서 아래로 실제로 읽고** 채운다.

| # | 확인할 것 | 판정 | 확인자 |
|---|---|---|---|
| 1 | **`#02` 층화 분할이 `#04` 로그변환 · `#05` 인코딩보다 위에 있다** — 셀 순서 자체가 증거다 | ✅ | 배지환 |
| 2 | `#04` 로그 변환이 train 통계를 쓰지 않는다 (`log_transform` 은 행 단위) | ✅ | 배지환 |
| 3 | `#05-3` 의 `reindex` 기준이 **train** 이다 (test 가 아니다) | ✅ | 배지환 |
| 4 | `#06` VIF 결과를 보고 **자동 제거를 하지 않았다** | ✅ | 배지환 |
| 5 | `#07` 의 B 가 A 와 **X 목록 한 줄만** 다르다 | ✅ | 배지환 |
| 6 | 기준 범주 교체 사유가 결정기록과 일치한다 (결정 23번) | ✅ | 배지환 |

**`#06` VIF 소견** — 임계 초과가 있었다면 무엇이 왜 올랐는지 한 문장으로 적는다.
자동 제거를 하지 않은 이유도 함께 적는다.

> ____________________________________________________________________

**분할 소견** — `#03` 의 세 비율을 직접 읽고 확인한 사람이 적는다.

> ____________________________________________________________________

**검수 종료 서명** — 만든 사람 : 공동   검수한 사람 : 배지환   일자 : 2026-09-01

---

## #09. 저장

| 파일 | 쓰임 |
|---|---|
| `12a_X_train.parquet` · `12b_X_test.parquet` | **⑬ 이 읽는 모형 A 설계행렬 26열** |
| `12c_y_train.csv` · `12d_y_test.csv` | `purchase_7d` |
| `12e_분할검증.csv` | 보고서 Ⅲ-1 — 층화가 걸렸다는 증거 3행 |
| `12f_왜도_전후.csv` | 보고서 Ⅲ-1 — 로그 변환의 효과 |
| `12g_더미후VIF.csv` | 보고서 Ⅲ-1 · Ⅳ-1 — 더미 전개 후 공선성 점검 |
| `12h_XB_train.parquet` · `12i_XB_test.parquet` | 결정 12번 민감도용 모형 B 84열 |
| `12j_기준범주.csv` | **⑧ 오즈비 해석이 읽는다** — "무엇 대비" 인가 |

CSV 는 타입을 보존하지 않으므로 설계행렬은 `parquet` 로 저장한다 (인덱스 `user_id` 보존).

In [15]:
assert not len(실패), "❌ #08-1 이 통과하지 않았다. 저장하지 않는다."

A_tr.to_parquet(OUT + r"\12a_X_train.parquet")
A_te.to_parquet(OUT + r"\12b_X_test.parquet")
y_tr.to_frame("purchase_7d").to_csv(OUT + r"\12c_y_train.csv", encoding="utf-8-sig")
y_te.to_frame("purchase_7d").to_csv(OUT + r"\12d_y_test.csv", encoding="utf-8-sig")

SPLIT.to_csv(OUT + r"\12e_분할검증.csv",   encoding="utf-8-sig")
SKEW.to_csv( OUT + r"\12f_왜도_전후.csv",  encoding="utf-8-sig")
VIF3.to_csv( OUT + r"\12g_더미후VIF.csv",  encoding="utf-8-sig")

B_tr.to_parquet(OUT + r"\12h_XB_train.parquet")
B_te.to_parquet(OUT + r"\12i_XB_test.parquet")

REFTAB.to_csv(OUT + r"\12j_기준범주.csv", encoding="utf-8-sig")

print("저장 완료")
print(f"  12a_X_train.parquet    {A_tr.shape[0]:,}행 × {A_tr.shape[1]}열")
print(f"  12b_X_test.parquet     {A_te.shape[0]:,}행 × {A_te.shape[1]}열")
print(f"  12c_y_train.csv        {len(y_tr):,}행   양성 {int(y_tr.sum()):,}명")
print(f"  12d_y_test.csv         {len(y_te):,}행   양성 {int(y_te.sum()):,}명")
print(f"  12e_분할검증.csv       {SPLIT.shape[0]}행")
print(f"  12f_왜도_전후.csv      {SKEW.shape[0]}행")
print(f"  12g_더미후VIF.csv      {VIF3.shape[0]}행   최대 {VIF3['VIF'].max():.3f}")
print(f"  12h_XB_train.parquet   {B_tr.shape[0]:,}행 × {B_tr.shape[1]}열")
print(f"  12i_XB_test.parquet    {B_te.shape[0]:,}행 × {B_te.shape[1]}열")
print(f"  12j_기준범주.csv       {REFTAB.shape[0]}행")

저장 완료
  12a_X_train.parquet    86,766행 × 26열
  12b_X_test.parquet     21,692행 × 26열
  12c_y_train.csv        86,766행   양성 17,789명
  12d_y_test.csv         21,692행   양성 4,447명
  12e_분할검증.csv       3행
  12f_왜도_전후.csv      3행
  12g_더미후VIF.csv      26행   최대 4.482
  12h_XB_train.parquet   86,766행 × 84열
  12i_XB_test.parquet    21,692행 × 84열
  12j_기준범주.csv       4행


---

## #10. 다음 — ⑬ 모델 구축

전처리를 여기서 끝냈으므로 `fit_pipeline` 에는 **적합만** 맡긴다.

```python
from pandas import read_parquet, read_csv
X_tr = read_parquet(OUT + r"\12a_X_train.parquet")
y_tr = read_csv(OUT + r"\12c_y_train.csv", encoding="utf-8-sig",
                index_col="user_id")["purchase_7d"]

fit = my_logit.fit_pipeline(concat([X_tr, y_tr], axis=1), y="purchase_7d",
                            encode=False, scale=False, vif=False, outlier=False)
```

| 할 일 | 어디서 | 비고 |
|---|---|---|
| `#08-2` 사람 확인 6개 · 소견 2개 · 서명 채우기 | **둘이 함께** | 저장 전에 |
| 기준선 로지스틱 적합 | `13` | `fit_pipeline(encode=False, scale=False)` |
| **가정 검정** — Box-Tidwell · VIF · 독립성 | `13` | LAB-10 05 |
| **결정 21번 판정** — `cart_product_view_cnt` 의 로짓 선형성 | `13` | 위배 시에만 제곱항 → 구간화 |
| **결정 12번 민감도** — 모형 B 로 핵심 X 3개의 부호·유의성 대조 | `13` | `12h`·`12i` 를 읽는다. **AUC 는 쓰지 않는다** |
| 성능 평가 — 혼동행렬 · F1 · **ROC-AUC** · 임계값 | `13` | `sklearn.metrics` 로 직접 작성 (helpers 에 분류 지표가 없다) |
| 5-Fold 교차검증 · Gap% | `13`/`14` | train 안에서 `StratifiedKFold` |

### 보고서 Ⅲ-1 에 반드시 옮길 것

| 옮길 것 | 어디서 |
|---|---|
| **분할이 가장 먼저였다는 사실** — 누수 등급표 | 표지 · `#02` |
| 층화 검증 3행 (전체 / train / test 가 20.50%) | `#03` · `12e` |
| 로그 변환 전후 왜도와 `prior_view_cnt` 가 1.0 을 넘는 한계 | `#04` · `12f` |
| **기준 범주를 지정한 사유** — 「교재 미기재」 표시 포함 | `#05-1` · `12j` |
| 더미 전개 후 VIF 와 자동 제거를 쓰지 않은 이유 | `#06` · `12g` |
| **구간화를 ⑬ 로 옮긴 사유와 대응 규칙** | 표지 · 결정 21번 |
| 스케일링을 하지 않은 사유 (βstd 는 원단위에서 계산된다) | 표지 |

> **⑤ 모델링용 전처리는 여기서 끝난다.** 다음 노트북부터 ⑥ 모델 구축이다.